In [ ]:
%pip install azure-monitor-query azure-identity pandas

In [1]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus
import pandas as pd
from datetime import timedelta

In [2]:
# DefaultAzureCredential tries multiple auth methods in order:
# 1. Environment variables, 2. Managed Identity, 3. Azure CLI, 4. VS Code, etc.
credential = DefaultAzureCredential()

# If that fails, use browser-based interactive login instead:
# credential = InteractiveBrowserCredential()

client = LogsQueryClient(credential)

In [3]:
WORKSPACE_ID = "aa95870a-82ef-4200-b64a-3cbb8e4967bd"

query = """
NTANetAnalytics
| where TimeGenerated > ago(5h)
| project TimeGenerated, SrcVm, SrcIp, DestVm, DestIp, DestPort, L7Protocol, FlowStatus, BytesSrcToDest, BytesDestToSrc
| order by TimeGenerated desc
"""

# query = """
# NTANetAnalytics
# | where TimeGenerated > ago(2h)
# | summarize count() by bin(TimeGenerated, 10m)
# | order by TimeGenerated desc
# """

response = client.query_workspace(
    workspace_id=WORKSPACE_ID,
    query=query,
    timespan=timedelta(hours=1)  # Safety net timespan; your query's ago(30m) will take precedence
)
 
if response.status == LogsQueryStatus.SUCCESS:
    table = response.tables[0]
    df = pd.DataFrame(data=table.rows, columns=table.columns)
    print(f"Returned {len(df)} rows")
    display(df)
else:
    print(f"Query failed: {response.partial_error}")

Returned 887 rows


,TimeGenerated,SrcVm,SrcIp,DestVm,DestIp,DestPort,L7Protocol,FlowStatus,BytesSrcToDest,BytesDestToSrc
0,2026-06-21 02:54:01.583708+00:00,,,,,NaN,,,NaN,NaN
1,2026-06-21 02:54:00.974750+00:00,,,rg-jxd-ddd/flowdemo-web2,10.0.1.4,6938.0,Unknown,Denied,0.0,0.0
2,2026-06-21 02:54:00.974750+00:00,,,rg-jxd-ddd/flowdemo-web2,10.0.1.4,56221.0,Unknown,Denied,0.0,0.0
3,2026-06-21 02:54:00.974750+00:00,,,rg-jxd-ddd/flowdemo-web2,10.0.1.4,9581.0,Unknown,Denied,0.0,0.0
4,2026-06-21 02:54:00.974750+00:00,,,rg-jxd-ddd/flowdemo-web2,10.0.1.4,8054.0,senomix03,Denied,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
882,2026-06-21 02:03:52.196660+00:00,,,rg-jxd-ddd/flowdemo-web2,10.0.1.4,3369.0,Unknown,Denied,0.0,0.0
883,2026-06-21 02:03:52.196660+00:00,,,rg-jxd-ddd/flowdemo-web2,10.0.1.4,21300.0,Unknown,Denied,0.0,0.0
884,2026-06-21 02:03:52.196660+00:00,,,rg-jxd-ddd/flowdemo-web2,10.0.1.4,2086.0,gnunet,Denied,0.0,0.0
885,2026-06-21 02:03:52.196660+00:00,,,rg-jxd-ddd/flowdemo-web2,10.0.1.4,2016.0,bootserver,Denied,0.0,0.0


In [ ]:
df.to_csv("flow_logs_data.csv", index=False)